# Fine-tuning do BERTimbau — Emi YouTube Analytics

**Ensaio da Sprint 1.** Os rótulos são os da Gemini (`rotulo_fraco`): as métricas de
validação daqui dizem se o *pipeline* funciona, **não** quanto o modelo acerta. O
número do Capítulo 5 sai de `ml/avaliacao/`, contra o `rotulo_humano`, depois que o
gabarito voltar.

Este notebook **não reimplementa nada**: ele clona o repositório e chama os mesmos
módulos que rodam localmente (`ml.treino.treinar`, `ml.treino.exportar_onnx`). Código
duplicado entre notebook e repositório diverge na primeira correção, e aí o modelo
publicado deixa de ser o que o repositório descreve.

**Antes de rodar:** Ambiente de execução → Alterar tipo de ambiente → **GPU (T4)**.

Fora do Colab ele roda igual, sem clonar nem instalar nada: é assim que
`ml/tests/test_notebook.py` executa todas as células num kernel, para provar que elas
funcionam antes de alguém gastar uma sessão de GPU com elas.

## 0. Onde tudo é gravado

Uma constante, e só uma: **`RAIZ`**. Todo caminho que o notebook grava ou baixa sai
dela — nenhuma célula monta caminho relativo por conta própria.

No **ensaio reduzido** (a variável `ENSAIO_REDUZIDO`, que `ml/tests/test_notebook.py`
usa para executar o notebook inteiro num kernel) todo arquivo ganha o sufixo
`-reduzido`. Um teste que passasse por cima dos pesos do treino de verdade custaria a
sessão de GPU que gerou aqueles pesos — e `ml/modelos/` está fora do git.

O motivo é um bug que custou uma sessão. O clone não era idempotente: rodado duas
vezes, ele criava `emi-youtube-analytics/emi-youtube-analytics`, e a partir daí cada
célula gravava numa cópia diferente conforme o diretório em que o kernel estivesse. O
treino salvou o modelo numa cópia, a exportação ONNX foi procurá-lo na outra — e morreu
com `FileNotFoundError: model_card.json`, sem gerar nem o `.onnx` nem o relatório.
`ml/modelos/` está inteiro no `.gitignore`, então a cópia nova nem essa pasta tinha.

In [ ]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

# NO_COLAB e a unica bifurcacao do notebook: clonar, instalar, ler o cofre e baixar
# arquivo sao coisas que so existem la. O resto do codigo e identico nos dois lados.
NO_COLAB = importlib.util.find_spec("google.colab") is not None

# CONSTANTE UNICA. Absoluta nos dois ambientes.
RAIZ = Path("/content/emi-youtube-analytics") if NO_COLAB else Path.cwd().resolve()
REPOSITORIO = "https://github.com/emi-youtube/emi-youtube-analytics.git"

ID_EXECUCAO = 4

# Ensaio REDUZIDO: corpus, grade e sementes cortados. E o modo em que
# ml/tests/test_notebook.py roda este arquivo inteiro num kernel — o caminho de codigo
# e o mesmo, so o tamanho muda. Nenhuma metrica daqui vale para nada.
REDUZIDO = bool(os.environ.get("ENSAIO_REDUZIDO"))
LIMITE = 150 if REDUZIDO else None
GRADE = {"taxas": (3e-5,), "epocas": (1,)} if REDUZIDO else {}

# Tudo o que o ensaio reduzido grava leva sufixo. Rodar o teste do notebook nao pode
# passar por cima dos pesos nem dos relatorios do treino de verdade: eles estao fora
# do git (ml/modelos/ inteiro esta no .gitignore) e uma sessao de T4 nao volta.
SUFIXO = "-reduzido" if REDUZIDO else ""

MODELO = RAIZ / "ml" / "modelos" / f"bertimbau-ensaio{SUFIXO}"
BUSCA = RAIZ / "ml" / "treino" / f"busca_hiperparametros{SUFIXO}.json"
SEMENTES_JSON = RAIZ / "ml" / "treino" / f"relatorio_sementes{SUFIXO}.json"
RELATORIO_ONNX = RAIZ / "ml" / "treino" / f"relatorio_onnx{SUFIXO}.json"
PREVISOES = RAIZ / "ml" / "dados" / f"previsoes_bertimbau{SUFIXO}.csv"
PACOTE = RAIZ / f"bertimbau-ensaio{SUFIXO}.zip"

print("colab:", NO_COLAB, "| raiz:", RAIZ, "| reduzido:", REDUZIDO)

## 1. Repositório e dependências

`ml/requirements-treino.txt` cobre tudo o que `ml/treino` importa: `asyncpg` (o banco),
`preprocessamento` (o `preparar_texto` que treino e inferência precisam executar
**identicamente** — CLAUDE.md Seção 3), `torch`, `transformers`, `onnx`, `onnxruntime`
e `psutil`.

Três detalhes do Colab, e os três já morderam:

- **o clone é idempotente.** Rodar a célula de novo atualiza o clone; nunca cria um
  segundo dentro do primeiro;
- **`!pip` que falha não interrompe o notebook.** O erro rola para fora da tela e a
  célula seguinte quebra com um `ModuleNotFoundError` que não tem nada a ver com a
  causa — foi assim que uma instalação incompleta apareceu como "No module named
  'asyncpg'" três células adiante. Por isso a instalação aqui é `subprocess.run(...,
  check=True)`: falhou, para na linha que falhou;
- **o `preprocessamento` é reinstalado sem `-e`.** A instalação editável registra um
  `.pth` que o interpretador só lê ao iniciar: num kernel que já está rodando, ela só
  valeria depois de *Reiniciar ambiente de execução*.

`INSTALACAO` é declarada como **dados**, e não como comando de shell, porque
`ml/tests/test_ambiente_colab.py` lê esta lista **deste arquivo** para montar um
ambiente limpo e conferir que ele importa todo o `ml/treino`. Copiar a lista de pacotes
para o teste criaria a segunda fonte de verdade que ele deveria detectar.

In [ ]:
INSTALACAO = (
    ["-q", "-r", "ml/requirements-treino.txt"],
    ["-q", "--force-reinstall", "--no-deps", "./preprocessamento"],
)

if NO_COLAB:
    if (RAIZ / ".git").exists():
        subprocess.run(["git", "-C", str(RAIZ), "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", REPOSITORIO, str(RAIZ)], check=True)

    for argumentos in INSTALACAO:
        # cwd=RAIZ: os caminhos de INSTALACAO sao relativos ao repositorio, e e a RAIZ
        # que diz qual repositorio e esse.
        subprocess.run(
            [sys.executable, "-m", "pip", "install", *argumentos], cwd=RAIZ, check=True
        )

os.chdir(RAIZ)
# Explicito, e nao herdado do diretorio de trabalho: depois do os.chdir, o que o
# interpretador ja tem em sys.path continua apontando para onde o kernel comecou.
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

print("raiz:", Path.cwd())

### Verificação do ambiente

Importa tudo o que as próximas células vão usar e confere as versões instaladas contra
os pinos do `requirements`. Se alguma coisa faltar ou divergir, o erro aparece **aqui**,
com o nome do que faltou — e não três células adiante.

A conferência de versão não é zelo excessivo: o Colab já vem com `torch` instalado, e
uma instalação que falhou em silêncio deixa o notebook rodando com a versão da casa em
vez da versão fixada. O treino até roda; quem quebra é a exportação ONNX, quatro células
depois, com uma mensagem que não menciona versão nenhuma.

In [ ]:
import importlib
from importlib.metadata import version

import preprocessamento
import torch

for modulo in (
    "asyncpg",
    "preprocessamento",
    "torch",
    "transformers",
    "onnx",
    "onnxruntime",
    "psutil",
    "numpy",
    "ml.treino.dados",
    "ml.treino.cartao",
    "ml.treino.treinar",
    "ml.treino.exportar_onnx",
    "ml.treino.prever_teste",
):
    importlib.import_module(modulo)

PINOS = {}
for arquivo in ("ml/requirements-treino.txt", "ml/requirements.txt"):
    for linha in (RAIZ / arquivo).read_text(encoding="utf-8").splitlines():
        limpa = linha.split("#")[0].strip()
        if "==" in limpa:
            nome, fixada = limpa.split("==")
            PINOS[nome.strip()] = fixada.strip()

# O sufixo local (2.5.1+cu124 no Colab, 2.5.1+cpu aqui) diz a variante de compilacao,
# nao a versao: o pino e a parte antes do "+".
divergentes = {
    nome: (fixada, version(nome))
    for nome, fixada in PINOS.items()
    if version(nome).split("+")[0] != fixada
}
assert not divergentes, f"versao instalada != pino do requirements: {divergentes}"

print("ambiente ok |", len(PINOS), "pacotes na versao fixada")
print("preprocessamento", preprocessamento.VERSAO)
print("torch", torch.__version__, "| GPU:", torch.cuda.is_available())

### A GPU

`nvidia-smi` só para registrar na saída qual placa a sessão pegou — o T4 é o que a conta
gratuita dá, e é o número do "custo de tempo" do README. Fora do Colab, a ausência do
comando não é erro.

In [ ]:
import shutil

if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
else:
    print("sem nvidia-smi: treino em CPU")

## 2. Credencial do banco

O `.env` **não está no repositório** (CLAUDE.md regra 1). Cole a `DATABASE_URL` no cofre
do Colab (🔑 no menu da esquerda) com o nome `DATABASE_URL`: assim ela não fica escrita
numa célula que o notebook salva junto com a saída.

Fora do Colab o `.env` já está na raiz do repositório, e esta célula não o toca.

In [ ]:
if NO_COLAB:
    from google.colab import userdata

    (RAIZ / ".env").write_text("DATABASE_URL=" + userdata.get("DATABASE_URL"), encoding="utf-8")

assert (RAIZ / ".env").exists(), f"{RAIZ / '.env'} nao existe — o banco nao vai abrir"
print(".env em", RAIZ / ".env", "(esta no .gitignore)")

## 3. Os dados

2.200 exemplos de `split IS NULL` com `rotulo_fraco`. Os 334 da amostra humana
(`split='teste'`) não entram — nem no treino, nem na validação.

A partição 85/15 é estratificada, com a semente do projeto (42), e **existe só em
memória**: nada é gravado no banco. Se o Kappa ficar abaixo de 0,60, a nova amostra
humana sai justamente destes 2.200, e marcá-los aqui esvaziaria esse pool.

**`await`, e não `asyncio.run`.** O kernel do Colab já tem um laço de eventos rodando, e
`asyncio.run` dentro de um laço vivo levanta `RuntimeError: asyncio.run() cannot be
called from a running event loop`. Em notebook a forma certa é o `await` de nível
superior; `asyncio.run` continua dentro do `treinar.py`, que é um script e tem o laço só
dele.

In [ ]:
import logging

from ml.treino.dados import (
    carregar_exemplos,
    dividir_estratificado,
    pesos_de_classe,
    relatar_particao,
)

logging.basicConfig(level="INFO", format="%(message)s", stream=sys.stdout, force=True)

exemplos = await carregar_exemplos(ID_EXECUCAO)
if LIMITE:
    exemplos = exemplos[:LIMITE]
    print(f"ENSAIO REDUZIDO: {LIMITE} exemplos. Nenhuma metrica daqui vale.")

particao = dividir_estratificado(exemplos)
relatar_particao(particao, pesos_de_classe(particao.treino))

## 4. Busca de hiperparâmetros — pela validação, e só

Nove configurações (3 taxas de aprendizado × 3 números de época). Cada uma treina do
zero, com **uma semente**, e é julgada pelo **F1 macro de validação**. O conjunto de
teste não aparece em nenhuma delas: olhar o teste e voltar para mexer na taxa de
aprendizado transformaria o teste num segundo conjunto de validação.

Repetir a grade com cinco sementes custaria a tarde de GPU inteira para escolher, quase
sempre, a mesma vencedora. A variação entre sementes é medida uma vez, na configuração
escolhida — é a célula 5.

Na T4 são ~2 min por configuração: reserve uns 25 minutos. Para pular a busca e treinar
uma configuração já escolhida, use o `Hiperparametros(...)` comentado dentro da célula.

In [ ]:
from ml.treino.treinar import buscar_hiperparametros, escolher_dispositivo, gravar_busca

dispositivo = escolher_dispositivo()
print("dispositivo:", dispositivo)

# Com a configuracao ja escolhida, pule a busca e diga qual e:
#   from ml.treino.treinar import Hiperparametros
#   escolhido = Hiperparametros(taxa_aprendizado=3e-5, epocas=3, lote=16)
escolhido, resultados_busca = buscar_hiperparametros(particao, dispositivo, **GRADE)
gravar_busca(resultados_busca, escolhido, BUSCA)

print()
print("escolhido pela validacao:", escolhido)

## 5. Treino oficial — cinco sementes, partição fixa

A configuração escolhida roda **cinco vezes**, mudando só a semente: a inicialização da
cabeça de classificação e a ordem dos lotes. A partição treino/validação é a mesma nas
cinco — se ela mudasse junto, o desvio padrão misturaria "o treino oscila" com "a
validação mudou", e não responderia nem uma coisa nem outra.

O que vai para o TCC é **média ± desvio padrão** do F1 macro de validação. O que vai
para o `backend/` é a **semente mediana**: publicar a melhor das cinco seria escolher
pelo máximo de uma amostra, e o artefato sairia com um número sistematicamente acima da
média reportada duas linhas antes.

As cinco rodadas ficam na memória (~420 MB cada) para que os pesos publicados sejam
exatamente os que produziram a linha mediana da tabela. Retreinar a mediana no fim seria
mais econômico e não reproduziria bit a bit em GPU.

In [ ]:
from ml.treino.treinar import (
    SEMENTES_OFICIAIS,
    escolher_semente_publicada,
    gravar_sementes,
    relatar_sementes,
    resumir_sementes,
    salvar_modelo,
    treinar_varias_sementes,
)

sementes = SEMENTES_OFICIAIS[:2] if REDUZIDO else SEMENTES_OFICIAIS
resultados = treinar_varias_sementes(particao, escolhido, dispositivo, sementes=sementes)

publicada = escolher_semente_publicada(resultados)
resumo = resumir_sementes(resultados, publicada)
relatar_sementes(resumo)
gravar_sementes(resumo, escolhido, SEMENTES_JSON)

salvar_modelo(publicada, particao, MODELO, sementes=resumo)

## 6. O `model_card.json`

O contrato com o `backend/`: `id2label` (a ordem dos rótulos sai daqui, nunca do código),
`max_length`, `versao_preprocessamento` (o worker recusa o modelo se divergir da versão
instalada), hiperparâmetros, a semente publicada e a tabela das cinco.

In [ ]:
import json

cartao = json.loads((MODELO / "model_card.json").read_text(encoding="utf-8"))
print(json.dumps(cartao, ensure_ascii=False, indent=2))

## 7. ONNX int8 — o modelo cabe no Azure B1?

A conversão e a medição rodam em **CPU**, com **uma thread**, porque é isso que a
instância B1 tem (1 vCPU, 1,75 GB). Medir na T4 responderia a pergunta errada.

Cada formato é medido num **processo separado**: com os três no mesmo processo, o RSS do
ONNX sairia somado ao do BERT do PyTorch ainda carregado.

Portão: perder no máximo **1 ponto de F1 macro** em relação ao original.

Os caminhos vão **absolutos**, a partir de `RAIZ`. Foi um caminho relativo daqui que,
resolvido no diretório errado, fez esta célula procurar o modelo numa cópia do
repositório que nunca tinha treinado nada.

In [ ]:
comando = [
    sys.executable,
    "-m",
    "ml.treino.exportar_onnx",
    "--id-execucao",
    str(ID_EXECUCAO),
    "--modelo",
    str(MODELO),
    "--relatorio",
    str(RELATORIO_ONNX),
]
if REDUZIDO:
    comando += ["--limite", "40"]

subprocess.run(comando, cwd=RAIZ, check=True)

assert RELATORIO_ONNX.exists(), f"{RELATORIO_ONNX} nao foi gerado"
assert (MODELO / "onnx" / "modelo_int8.onnx").exists(), "o grafo int8 nao foi gerado"
print("ok:", RELATORIO_ONNX)

## 8. Levar os artefatos embora

O Colab apaga a máquina quando a sessão termina. Baixe a pasta do modelo (pesos,
tokenizer, `model_card.json` e os dois grafos ONNX) e os três relatórios versionáveis.

A conferência de existência roda **fora do Colab também**: é ela que faz o teste do
notebook falhar quando uma célula anterior deixou de gravar o que prometia.

In [ ]:
ARTEFATOS = (PACOTE, BUSCA, SEMENTES_JSON, RELATORIO_ONNX)

PACOTE.unlink(missing_ok=True)
shutil.make_archive(str(PACOTE.with_suffix("")), "zip", MODELO.parent, MODELO.name)

for arquivo in ARTEFATOS:
    assert arquivo.exists(), f"{arquivo} nao existe — a celula que o gera falhou?"
    print(f"{arquivo.stat().st_size / 1_048_576:8.1f} MB  {arquivo}")

if NO_COLAB:
    from google.colab import files

    for arquivo in ARTEFATOS:
        files.download(str(arquivo))

## 9. O conjunto de teste — uma única vez, no fim

`prever_teste.py` classifica os 334 **sem ler nenhum rótulo** e grava um CSV. Nenhuma
métrica sai dele: a comparação contra o gabarito humano é `ml/avaliacao/avaliar.py`, um
processo separado, rodado depois — e que só faz sentido quando o `rotulo_humano` já
existir.

Enquanto o gabarito não voltar, esta célula produz previsões que ficam esperando.

In [ ]:
subprocess.run(
    [
        sys.executable,
        "-m",
        "ml.treino.prever_teste",
        "--id-execucao",
        str(ID_EXECUCAO),
        "--modelo",
        str(MODELO),
        "--saida",
        str(PREVISOES),
    ],
    cwd=RAIZ,
    check=True,
)

assert PREVISOES.exists(), f"{PREVISOES} nao foi gerado"
print("previsoes em", PREVISOES)

if NO_COLAB:
    files.download(str(PREVISOES))

# Depois que o gabarito voltar, localmente, com as previsoes em maos:
#   python -m ml.avaliacao.avaliar --id-execucao 4 --gemini
#     --previsoes bertimbau=ml/dados/previsoes_bertimbau.csv
#     --previsoes lexico=ml/dados/previsoes_lexico.csv